In [2]:
import os
import sys

java_home = r"C:\Program Files\Java\jdk-17"

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Week6-Superstore")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Spark Version:", spark.version)

JAVA_HOME: C:\Program Files\Java\jdk-17
Spark Version: 4.1.2


In [3]:
df = spark.read.csv(
    r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Data\Sample - Superstore.csv",   
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True,
)

print("Row count:", df.count())
df.printSchema()
df.show(5)

Row count: 9994
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+

In [4]:
 # Q1 — Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.
# - Driver: runs the main program, builds the DAG, schedules tasks.
# - Cluster Manager: allocates resources (cores/memory) across the cluster — e.g. YARN, Kubernetes, Spark Standalone.
# - Executor: runs on worker nodes, executes tasks assigned by the driver, stores data for caching.


In [5]:
 #Q2 —  How does Spark’s Lazy Evaluation strategy improve performance when chain
#processing large datasets? 
#  Spark doesn't run transformations immediately —
# it builds a logical plan (DAG) and only executes when an action is called, letting the
# optimizer combine/reorder steps and skip unnecessary work.


In [6]:
#Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring 
#the first row is treated as a header and inferSchema is enabled.

 #As asked in the question (generic path)
# df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
 
# Applied to our real file:
df_q3 = spark.read.csv(
   r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Data\Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
)
df_q3.show(3)


+------+--------------+----------+----------+------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|Row ID|      Order ID|Order Date| Ship Date|   Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|Second Class|   CG-12520|    Claire Gute| Consumer|United States|  Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|261.96|       2|     0.0|4

In [17]:
import os
import csv

base_folder = r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment"
output_folder = os.path.join(base_folder, "Output")
os.makedirs(output_folder, exist_ok=True)

csv_path = os.path.join(output_folder, "superstore_copy.csv")
parquet_path = os.path.join(output_folder, "superstore_copy.parquet")

# Save CSV (row-based)
rows = df.collect()
columns = df.columns

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    writer.writerows(rows)

# Save Parquet (columnar)
pandas_df = df.toPandas()
pandas_df.to_parquet(parquet_path, engine="fastparquet", index=False)

# File sizes
csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)

print("CSV saved to:", csv_path)
print("Parquet saved to:", parquet_path)
print()
print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {parquet_size:,} bytes")
print(f"Parquet is {csv_size/parquet_size:.2f}x smaller")

CSV saved to: C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\superstore_copy.csv
Parquet saved to: C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\superstore_copy.parquet

CSV size:     2,299,053 bytes
Parquet size: 817,881 bytes
Parquet is 2.81x smaller


In [18]:
#Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.
result_q5 = df.filter(df.Category == "Technology").select("Product ID", "Sales")
result_q5.show(5)

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
+---------------+--------+
only showing top 5 rows


In [19]:
#Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double
df_q6 = (
    df
    .withColumnRenamed("Sub-Category", "SubCategory")
    .withColumn("Sales", F.col("Sales").cast(DoubleType()))
)
df_q6.select("SubCategory", "Sales").show(5)
df_q6.printSchema()

+-----------+--------+
|SubCategory|   Sales|
+-----------+--------+
|  Bookcases|  261.96|
|     Chairs|  731.94|
|     Labels|   14.62|
|     Tables|957.5775|
|    Storage|  22.368|
+-----------+--------+
only showing top 5 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable =

In [20]:
#Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 
#Spark tracks the sequence of transformations
# (lineage) that built each RDD/DataFrame partition. If a worker node fails, Spark
# recomputes only the lost partitions using that lineage, instead of restarting the
# whole job.

In [23]:
#Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 
demo_schema_q8 = StructType([
    StructField("order_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount", DoubleType(), True),
])
demo_data_q8 = [
    ("O1", "Completed", 1500.0),
    ("O2", "Pending", 2000.0),
    ("O3", "Completed", 500.0),
    ("O4", "Completed", 1200.0),
]
df_orders = spark.createDataFrame(demo_data_q8, demo_schema_q8)

result_q8 = df_orders.filter((df_orders.status == "Completed") & (df_orders.amount > 1000))
result_q8.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|      O1|Completed|1500.0|
|      O4|Completed|1200.0|
+--------+---------+------+



In [25]:
#Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.
#Demonstrated below: filtering the Parquet file
# we wrote in Q4 and checking the physical plan — the `PushedFilters` line confirms
# the filter is applied at the storage layer, before data is loaded into memory
df_parquet = spark.read.parquet(r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\superstore_copy.parquet")
df_parquet.filter(df_parquet.Category == "Technology").explain()

== Physical Plan ==
*(1) Filter (isnotnull(Category#384) AND (Category#384 = Technology))
+- *(1) ColumnarToRow
   +- FileScan parquet [Row ID#370,Order ID#371,Order Date#372,Ship Date#373,Ship Mode#374,Customer ID#375,Customer Name#376,Segment#377,Country#378,City#379,State#380,Postal Code#381,Region#382,Product ID#383,Category#384,Sub-Category#385,Product Name#386,Sales#387,Quantity#388,Discount#389,Profit#390] Batched: true, DataFilters: [isnotnull(Category#384), (Category#384 = Technology)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Varad/Desktop/Celebal/Week6_Spark_assignment/Output/sup..., PartitionFilters: [], PushedFilters: [IsNotNull(Category), EqualTo(Category,Technology)], ReadSchema: struct<Row ID:int,Order ID:string,Order Date:string,Ship Date:string,Ship Mode:string,Customer ID...




In [26]:
#Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 
df_q10 = df.withColumn("final_price", F.col("Sales") * 1.18)
df_q10.select("Sales", "final_price").show(5)

+--------+------------------+
|   Sales|       final_price|
+--------+------------------+
|  261.96|309.11279999999994|
|  731.94|          863.6892|
|   14.62|           17.2516|
|957.5775|        1129.94145|
|  22.368|26.394239999999996|
+--------+------------------+
only showing top 5 rows


In [27]:
#Q11: What is the difference between Transformations and Actions? Provide two examples of each.
#transformations (e.g. `filter`, `select`) are lazy and return a new DataFrame; actions (e.g. `count`, `show`) trigger actual
# execution and return a result.

In [28]:
#Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

df_loaded = spark.read.parquet(parquet_path)
df_no_nulls = df_loaded.filter(df_loaded["Customer ID"].isNotNull())

# Save as CSV (via collect + csv module, since Spark's CSV writer needs winutils.exe here)
rows_out = df_no_nulls.collect()
cols_out = df_no_nulls.columns

with open(r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\q12_output.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(cols_out)
    for r in rows_out:
        w.writerow(r)

print("Rows after filtering null Customer ID:", df_no_nulls.count())
print("Saved to:", r"C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\q12_output.csv")

Rows after filtering null Customer ID: 9994
Saved to: C:\Users\Varad\Desktop\Celebal\Week6_Spark_assignment\Output\q12_output.csv


In [29]:
#Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?
#in Client Mode the Driver runs on the machine that submitted the job (outside the cluster); in Cluster Mode the Driver runs
# inside the cluster itself, managed by the Cluster Manager.

In [30]:
#Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 
result_q14 = df.filter((df.Region == "North") | (df["Ship Mode"] == "First Class"))
result_q14.select("Region", "Ship Mode", "Customer Name").show(5)
print("Matching rows:", result_q14.count())

+-------+-----------+---------------+
| Region|  Ship Mode|  Customer Name|
+-------+-----------+---------------+
|Central|First Class|      Gene Hale|
|Central|First Class|      Gene Hale|
|Central|First Class|  Odella Nelson|
|Central|First Class|  Odella Nelson|
|   East|First Class|Ted Butterfield|
+-------+-----------+---------------+
only showing top 5 rows
Matching rows: 1538


In [31]:
#Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?
#collect()` pulls **every row** back to the
# driver's memory, which can crash the driver on huge datasets. `.show(5)` only
# computes and displays a small sample, keeping memory usage minimal.